<a href="https://colab.research.google.com/github/husthorng/Backpropagation_NN/blob/main/2026_MACHINE_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================================
# DHT_LDR / tr33
#
# Neural Network 3 - 13 - 3
#
# 第 1~3 欄 = Input
# 第 4~6 欄 = Target Output
#
# Input  Layer  = 3
# Hidden Layer  = 13
# Output Layer  = 3
# ==========================================================

# ==========================================================
# 1. 匯入套件
# ==========================================================

!pip install -q gspread

import numpy as np
import pandas as pd
import gspread

from google.colab import auth
from google.auth import default


# ==========================================================
# 2. Google 授權
# ==========================================================

auth.authenticate_user()

creds, _ = default()
gc = gspread.authorize(creds)


# ==========================================================
# 3. 開啟 Google Sheet
# ==========================================================

SPREADSHEET_NAME = "DHT_LDR"
WORKSHEET_NAME = "tr33"

sh = gc.open(SPREADSHEET_NAME)
ws = sh.worksheet(WORKSHEET_NAME)


# ==========================================================
# 4. 讀取 tr33 全部資料
# ==========================================================

values = ws.get_all_values()

# 第一列當欄位名稱
header = values[0]

# 第二列開始為資料
data = values[1:]

df = pd.DataFrame(data, columns=header)

print("==========================================")
print("Google Sheet 讀取完成")
print("==========================================")

print("工作表 :", WORKSHEET_NAME)
print("資料筆數 :", len(df))
print("欄位數 :", len(df.columns))

print("\n欄位名稱：")
for i, col in enumerate(df.columns):
    print(f"第 {i+1} 欄 : {col}")

print("\n前 10 筆資料：")
display(df.head(10))



Google Sheet 讀取完成
工作表 : tr33
資料筆數 : 7
欄位數 : 6

欄位名稱：
第 1 欄 : PB1
第 2 欄 : PB2
第 3 欄 : PB3
第 4 欄 : TL1
第 5 欄 : TL2
第 6 欄 : TL3

前 10 筆資料：


,PB1,PB2,PB3,TL1,TL2,TL3
0,0,0,0,0,0,1
1,1,0,0,0,1,0
2,0,1,0,0,0,1
3,0,0,1,0,0,1
4,1,1,0,1,0,0
5,0,1,1,0,0,1
6,1,1,1,1,0,0


In [2]:

# ==========================================================
# 5. 只取前 6 欄
#
# 第 1~3 欄 = Input
# 第 4~6 欄 = Target
# ==========================================================

if len(df.columns) < 6:
    raise ValueError(
        f"資料欄位只有 {len(df.columns)} 欄，必須至少有 6 欄！"
    )

input_columns = df.columns[:3]
target_columns = df.columns[3:6]

print("\n==========================================")
print("神經網路資料設定")
print("==========================================")

print("Input 欄位：")
print(list(input_columns))

print("\nTarget 欄位：")
print(list(target_columns))


# ==========================================================
# 6. 將資料轉成數值
# ==========================================================

X = df.iloc[:, 0:3].apply(
    pd.to_numeric,
    errors="coerce"
)

Y = df.iloc[:, 3:6].apply(
    pd.to_numeric,
    errors="coerce"
)


# ==========================================================
# 7. 移除空白 / 非數字資料
# ==========================================================

valid = ~(X.isna().any(axis=1) | Y.isna().any(axis=1))

X = X[valid]
Y = Y[valid]

X = X.to_numpy(dtype=np.float64)
Y = Y.to_numpy(dtype=np.float64)


print("\n==========================================")
print("有效資料")
print("==========================================")

print("有效筆數 :", len(X))

print("\nInput X shape :", X.shape)
print("Target Y shape:", Y.shape)

print("\nInput 前 5 筆：")
print(X[:5])

print("\nTarget 前 5 筆：")
print(Y[:5])




神經網路資料設定
Input 欄位：
['PB1', 'PB2', 'PB3']

Target 欄位：
['TL1', 'TL2', 'TL3']

有效資料
有效筆數 : 7

Input X shape : (7, 3)
Target Y shape: (7, 3)

Input 前 5 筆：
[[0. 0. 0.]
 [1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [1. 1. 0.]]

Target 前 5 筆：
[[0. 0. 1.]
 [0. 1. 0.]
 [0. 0. 1.]
 [0. 0. 1.]
 [1. 0. 0.]]


In [3]:

# ==========================================================
# 8. Input 正規化
#
# 避免溫度、濕度、LDR數值差距太大
#
# 使用 Min-Max：
#
# X_norm = (X - min) / (max - min)
# ==========================================================

X_min = X.min(axis=0)
X_max = X.max(axis=0)

X_range = X_max - X_min

# 避免某一欄全部都是相同數值
X_range[X_range == 0] = 1

X_norm = (X - X_min) / X_range


# ==========================================================
# 9. Target 正規化
# ==========================================================

Y_min = Y.min(axis=0)
Y_max = Y.max(axis=0)

Y_range = Y_max - Y_min

Y_range[Y_range == 0] = 1

Y_norm = (Y - Y_min) / Y_range


In [4]:

# ==========================================================
# 10. 神經網路設定
# ==========================================================

INPUT_SIZE = 3
HIDDEN_SIZE = 13
OUTPUT_SIZE = 3

print("\n==========================================")
print("Neural Network")
print("==========================================")

print(
    f"Input Layer  : {INPUT_SIZE}"
)

print(
    f"Hidden Layer : {HIDDEN_SIZE}"
)

print(
    f"Output Layer : {OUTPUT_SIZE}"
)


# ==========================================================
# 11. Sigmoid
# ==========================================================

def sigmoid(x):

    # 防止 exp overflow
    x = np.clip(x, -50, 50)

    return 1.0 / (1.0 + np.exp(-x))


# ==========================================================
# 12. Sigmoid derivative
# ==========================================================

def sigmoid_derivative(x):

    s = sigmoid(x)

    return s * (1.0 - s)


# ==========================================================
# 13. 建立權重
#
# W0 : 3 × 13
# W1 : 13 × 3
#
# Bias：
# B0 : 13
# B1 : 3
# ==========================================================

np.random.seed(42)

W0 = np.random.uniform(
    -1,
    1,
    (INPUT_SIZE, HIDDEN_SIZE)
)

B0 = np.zeros(HIDDEN_SIZE)

W1 = np.random.uniform(
    -1,
    1,
    (HIDDEN_SIZE, OUTPUT_SIZE)
)

B1 = np.zeros(OUTPUT_SIZE)


# ==========================================================
# 14. Forward
# ==========================================================

def forward(X):

    hidden_input = np.dot(X, W0) + B0

    hidden_output = sigmoid(hidden_input)

    output_input = np.dot(hidden_output, W1) + B1

    output = sigmoid(output_input)

    return (
        hidden_input,
        hidden_output,
        output_input,
        output
    )


# ==========================================================
# 15. 訓練參數
# ==========================================================

LEARNING_RATE = 0.5

EPOCHS = 100000

print("\n==========================================")
print("開始訓練")
print("==========================================")

print("Learning Rate :", LEARNING_RATE)
print("Epochs        :", EPOCHS)




Neural Network
Input Layer  : 3
Hidden Layer : 13
Output Layer : 3

開始訓練
Learning Rate : 0.5
Epochs        : 100000


In [5]:

# ==========================================================
# 16. Backpropagation
# ==========================================================

for epoch in range(EPOCHS):

    # --------------------------------------
    # Forward
    # --------------------------------------

    hidden_input, hidden_output, output_input, output = \
        forward(X_norm)


    # --------------------------------------
    # Error
    # --------------------------------------

    error = Y_norm - output


    # --------------------------------------
    # Output layer gradient
    # --------------------------------------

    output_delta = (
        error *
        sigmoid_derivative(output_input)
    )


    # --------------------------------------
    # Hidden layer gradient
    # --------------------------------------

    hidden_delta = (
        np.dot(output_delta, W1.T) *
        sigmoid_derivative(hidden_input)
    )


    # --------------------------------------
    # 更新 W1
    # --------------------------------------

    W1 += (
        np.dot(hidden_output.T, output_delta)
        * LEARNING_RATE
    )


    # --------------------------------------
    # 更新 B1
    # --------------------------------------

    B1 += (
        np.sum(output_delta, axis=0)
        * LEARNING_RATE
    )


    # --------------------------------------
    # 更新 W0
    # --------------------------------------

    W0 += (
        np.dot(X_norm.T, hidden_delta)
        * LEARNING_RATE
    )


    # --------------------------------------
    # 更新 B0
    # --------------------------------------

    B0 += (
        np.sum(hidden_delta, axis=0)
        * LEARNING_RATE
    )


    # --------------------------------------
    # 每 5000 次顯示 Loss
    # --------------------------------------

    if epoch % 5000 == 0:

        loss = np.mean(error ** 2)

        print(
            f"Epoch {epoch:6d} "
            f"Loss = {loss:.10f}"
        )


# ==========================================================
# 17. 最終 Loss
# ==========================================================

_, _, _, prediction_norm = forward(X_norm)

loss = np.mean(
    (Y_norm - prediction_norm) ** 2
)

print("\n==========================================")
print("訓練完成")
print("==========================================")

print("Final Loss =", loss)



Epoch      0 Loss = 0.3228961009
Epoch   5000 Loss = 0.0000566945
Epoch  10000 Loss = 0.0000261478
Epoch  15000 Loss = 0.0000167525
Epoch  20000 Loss = 0.0000122468
Epoch  25000 Loss = 0.0000096171
Epoch  30000 Loss = 0.0000078994
Epoch  35000 Loss = 0.0000066920
Epoch  40000 Loss = 0.0000057982
Epoch  45000 Loss = 0.0000051107
Epoch  50000 Loss = 0.0000045659
Epoch  55000 Loss = 0.0000041239
Epoch  60000 Loss = 0.0000037582
Epoch  65000 Loss = 0.0000034508
Epoch  70000 Loss = 0.0000031890
Epoch  75000 Loss = 0.0000029633
Epoch  80000 Loss = 0.0000027667
Epoch  85000 Loss = 0.0000025942
Epoch  90000 Loss = 0.0000024414
Epoch  95000 Loss = 0.0000023053

訓練完成
Final Loss = 2.1832346628410557e-06


In [6]:

# ==========================================================
# 18. 顯示預測結果
# ==========================================================

prediction = (
    prediction_norm * Y_range
    + Y_min
)

print("\n==========================================")
print("實際 Target / 預測 Output")
print("==========================================")

result = pd.DataFrame(
    np.hstack((Y, prediction)),
    columns=[
        "Target1",
        "Target2",
        "Target3",
        "Predict1",
        "Predict2",
        "Predict3"
    ]
)

display(result.head(20))


# ==========================================================
# 19. 顯示 W0
# ==========================================================

print("\n==========================================")
print("W0  (3 × 13)")
print("==========================================")

print(W0)


# ==========================================================
# 20. 顯示 B0
# ==========================================================

print("\n==========================================")
print("B0  (13)")
print("==========================================")

print(B0)


# ==========================================================
# 21. 顯示 W1
# ==========================================================

print("\n==========================================")
print("W1  (13 × 3)")
print("==========================================")

print(W1)


# ==========================================================
# 22. 顯示 B1
# ==========================================================

print("\n==========================================")
print("B1  (3)")
print("==========================================")

print(B1)



實際 Target / 預測 Output


,Target1,Target2,Target3,Predict1,Predict2,Predict3
0,0.0,0.0,1.0,5.904601e-07,1.666531e-03,0.999144
1,0.0,1.0,0.0,2.340217e-03,9.974527e-01,0.000950
2,0.0,0.0,1.0,1.549103e-03,1.521011e-06,0.998557
3,0.0,0.0,1.0,9.849254e-06,3.757578e-05,0.999345
4,1.0,0.0,0.0,9.972767e-01,2.265720e-03,0.001558
5,0.0,0.0,1.0,1.368241e-03,8.368419e-07,0.998854
6,1.0,0.0,0.0,9.984094e-01,7.710600e-04,0.001809



W0  (3 × 13)
[[-2.88381502  2.60038253  1.4249018  -3.13432407 -3.63909928  0.94610976
  -4.1188203  -1.58537104 -2.47121387  1.86641641 -0.23743273  1.39263153
   3.88191376]
 [-0.25417667 -4.53079618  1.10198012  0.02105468 -0.37813662 -2.68040202
  -0.167477    3.66307177 -0.73932442 -2.82749539 -0.79216749 -4.00971312
   1.50587597]
 [-0.03914892 -0.84108311  0.28244242 -0.2146609   0.29615711 -1.14790337
  -0.02056521  1.44924772  0.87529346  0.19163098 -0.68989163 -1.32104167
  -0.05003804]]

B0  (13)
[ 1.32540579 -0.66059124 -0.95744832  1.28613488  1.62826671  0.0532066
  1.95011689  0.10595644  1.03435021 -0.56555786 -0.32968245  0.26297175
 -3.14849395]

W1  (13 × 3)
[[-2.62716279 -1.8926369   1.85998474]
 [-3.74805699  4.52873201 -1.42877758]
 [ 2.14610499 -1.11649477 -2.04496717]
 [-2.18615823 -1.95833175  2.67632564]
 [-2.71161024 -1.10648623  3.75916605]
 [-2.00180685  2.22121146 -0.50977936]
 [-4.05879061 -2.83563163  2.69224772]
 [ 2.62111632 -4.43281885 -0.3408852 ]
 

In [7]:

# ==========================================================
# 23. 輸出 ESP32 C++ 格式
# ==========================================================

def print_cpp_matrix(name, matrix):

    rows, cols = matrix.shape

    print(f"float {name}[{rows}][{cols}] = {{")

    for row in matrix:

        print(
            "    {"
            + ", ".join(f"{v:.10f}" for v in row)
            + "},"
        )

    print("};")
    print()


def print_cpp_array(name, array):

    print(
        f"float {name}[{len(array)}] = {{"
    )

    print(
        "    "
        + ", ".join(f"{v:.10f}" for v in array)
    )

    print("};")
    print()


print("\n")
print("==========================================================")
print("ESP32 C++ WEIGHTS")
print("==========================================================")

print_cpp_matrix("W0", W0)

print_cpp_array("B0", B0)

print_cpp_matrix("W1", W1)

print_cpp_array("B1", B1)


# ==========================================================
# 24. 正規化參數
#
# ESP32 如果要使用相同的神經網路，
# 必須同時使用這些參數。
# ==========================================================

print("\n")
print("==========================================================")
print("ESP32 NORMALIZATION PARAMETERS")
print("==========================================================")

print_cpp_array("X_MIN", X_min)

print_cpp_array("X_MAX", X_max)

print_cpp_array("Y_MIN", Y_min)

print_cpp_array("Y_MAX", Y_max)



ESP32 C++ WEIGHTS
float W0[3][13] = {
    {-2.8838150217, 2.6003825284, 1.4249018044, -3.1343240685, -3.6390992765, 0.9461097598, -4.1188202980, -1.5853710433, -2.4712138672, 1.8664164143, -0.2374327322, 1.3926315253, 3.8819137578},
    {-0.2541766705, -4.5307961844, 1.1019801223, 0.0210546816, -0.3781366168, -2.6804020208, -0.1674770049, 3.6630717674, -0.7393244158, -2.8274953928, -0.7921674897, -4.0097131195, 1.5058759660},
    {-0.0391489205, -0.8410831120, 0.2824424194, -0.2146609026, 0.2961571059, -1.1479033692, -0.0205652087, 1.4492477195, 0.8752934640, 0.1916309833, -0.6898916340, -1.3210416740, -0.0500380412},
};

float B0[13] = {
    1.3254057940, -0.6605912391, -0.9574483175, 1.2861348817, 1.6282667056, 0.0532066003, 1.9501168857, 0.1059564405, 1.0343502055, -0.5655578639, -0.3296824472, 0.2629717460, -3.1484939495
};

float W1[13][3] = {
    {-2.6271627887, -1.8926368975, 1.8599847437},
    {-3.7480569912, 4.5287320058, -1.4287775755},
    {2.1461049889, -1.1164947749, -2.

In [ ]:
# ==========================================================
# DHT_LDR / tr33
#
# Neural Network 3 - 13 - 3
#
# 第 1~3 欄 = Input
# 第 4~6 欄 = Target Output
#
# Input  Layer  = 3
# Hidden Layer  = 13
# Output Layer  = 3
# ==========================================================

# ==========================================================
# 1. 匯入套件
# ==========================================================

!pip install -q gspread

import numpy as np
import pandas as pd
import gspread

from google.colab import auth
from google.auth import default


# ==========================================================
# 2. Google 授權
# ==========================================================

auth.authenticate_user()

creds, _ = default()
gc = gspread.authorize(creds)


# ==========================================================
# 3. 開啟 Google Sheet
# ==========================================================

SPREADSHEET_NAME = "DHT_LDR"
WORKSHEET_NAME = "tr33"

sh = gc.open(SPREADSHEET_NAME)
ws = sh.worksheet(WORKSHEET_NAME)


# ==========================================================
# 4. 讀取 tr33 全部資料
# ==========================================================

values = ws.get_all_values()

# 第一列當欄位名稱
header = values[0]

# 第二列開始為資料
data = values[1:]

df = pd.DataFrame(data, columns=header)

print("==========================================")
print("Google Sheet 讀取完成")
print("==========================================")

print("工作表 :", WORKSHEET_NAME)
print("資料筆數 :", len(df))
print("欄位數 :", len(df.columns))

print("\n欄位名稱：")
for i, col in enumerate(df.columns):
    print(f"第 {i+1} 欄 : {col}")

print("\n前 10 筆資料：")
display(df.head(10))


# ==========================================================
# 5. 只取前 6 欄
#
# 第 1~3 欄 = Input
# 第 4~6 欄 = Target
# ==========================================================

if len(df.columns) < 6:
    raise ValueError(
        f"資料欄位只有 {len(df.columns)} 欄，必須至少有 6 欄！"
    )

input_columns = df.columns[:3]
target_columns = df.columns[3:6]

print("\n==========================================")
print("神經網路資料設定")
print("==========================================")

print("Input 欄位：")
print(list(input_columns))

print("\nTarget 欄位：")
print(list(target_columns))


# ==========================================================
# 6. 將資料轉成數值
# ==========================================================

X = df.iloc[:, 0:3].apply(
    pd.to_numeric,
    errors="coerce"
)

Y = df.iloc[:, 3:6].apply(
    pd.to_numeric,
    errors="coerce"
)


# ==========================================================
# 7. 移除空白 / 非數字資料
# ==========================================================

valid = ~(X.isna().any(axis=1) | Y.isna().any(axis=1))

X = X[valid]
Y = Y[valid]

X = X.to_numpy(dtype=np.float64)
Y = Y.to_numpy(dtype=np.float64)


print("\n==========================================")
print("有效資料")
print("==========================================")

print("有效筆數 :", len(X))

print("\nInput X shape :", X.shape)
print("Target Y shape:", Y.shape)

print("\nInput 前 5 筆：")
print(X[:5])

print("\nTarget 前 5 筆：")
print(Y[:5])


# ==========================================================
# 8. Input 正規化
#
# 避免溫度、濕度、LDR數值差距太大
#
# 使用 Min-Max：
#
# X_norm = (X - min) / (max - min)
# ==========================================================

X_min = X.min(axis=0)
X_max = X.max(axis=0)

X_range = X_max - X_min

# 避免某一欄全部都是相同數值
X_range[X_range == 0] = 1

X_norm = (X - X_min) / X_range


# ==========================================================
# 9. Target 正規化
# ==========================================================

Y_min = Y.min(axis=0)
Y_max = Y.max(axis=0)

Y_range = Y_max - Y_min

Y_range[Y_range == 0] = 1

Y_norm = (Y - Y_min) / Y_range


# ==========================================================
# 10. 神經網路設定
# ==========================================================

INPUT_SIZE = 3
HIDDEN_SIZE = 13
OUTPUT_SIZE = 3

print("\n==========================================")
print("Neural Network")
print("==========================================")

print(
    f"Input Layer  : {INPUT_SIZE}"
)

print(
    f"Hidden Layer : {HIDDEN_SIZE}"
)

print(
    f"Output Layer : {OUTPUT_SIZE}"
)


# ==========================================================
# 11. Sigmoid
# ==========================================================

def sigmoid(x):

    # 防止 exp overflow
    x = np.clip(x, -50, 50)

    return 1.0 / (1.0 + np.exp(-x))


# ==========================================================
# 12. Sigmoid derivative
# ==========================================================

def sigmoid_derivative(x):

    s = sigmoid(x)

    return s * (1.0 - s)


# ==========================================================
# 13. 建立權重
#
# W0 : 3 × 13
# W1 : 13 × 3
#
# Bias：
# B0 : 13
# B1 : 3
# ==========================================================

np.random.seed(42)

W0 = np.random.uniform(
    -1,
    1,
    (INPUT_SIZE, HIDDEN_SIZE)
)

B0 = np.zeros(HIDDEN_SIZE)

W1 = np.random.uniform(
    -1,
    1,
    (HIDDEN_SIZE, OUTPUT_SIZE)
)

B1 = np.zeros(OUTPUT_SIZE)


# ==========================================================
# 14. Forward
# ==========================================================

def forward(X):

    hidden_input = np.dot(X, W0) + B0

    hidden_output = sigmoid(hidden_input)

    output_input = np.dot(hidden_output, W1) + B1

    output = sigmoid(output_input)

    return (
        hidden_input,
        hidden_output,
        output_input,
        output
    )


# ==========================================================
# 15. 訓練參數
# ==========================================================

LEARNING_RATE = 0.5

EPOCHS = 100000

print("\n==========================================")
print("開始訓練")
print("==========================================")

print("Learning Rate :", LEARNING_RATE)
print("Epochs        :", EPOCHS)


# ==========================================================
# 16. Backpropagation
# ==========================================================

for epoch in range(EPOCHS):

    # --------------------------------------
    # Forward
    # --------------------------------------

    hidden_input, hidden_output, output_input, output = \
        forward(X_norm)


    # --------------------------------------
    # Error
    # --------------------------------------

    error = Y_norm - output


    # --------------------------------------
    # Output layer gradient
    # --------------------------------------

    output_delta = (
        error *
        sigmoid_derivative(output_input)
    )


    # --------------------------------------
    # Hidden layer gradient
    # --------------------------------------

    hidden_delta = (
        np.dot(output_delta, W1.T) *
        sigmoid_derivative(hidden_input)
    )


    # --------------------------------------
    # 更新 W1
    # --------------------------------------

    W1 += (
        np.dot(hidden_output.T, output_delta)
        * LEARNING_RATE
    )


    # --------------------------------------
    # 更新 B1
    # --------------------------------------

    B1 += (
        np.sum(output_delta, axis=0)
        * LEARNING_RATE
    )


    # --------------------------------------
    # 更新 W0
    # --------------------------------------

    W0 += (
        np.dot(X_norm.T, hidden_delta)
        * LEARNING_RATE
    )


    # --------------------------------------
    # 更新 B0
    # --------------------------------------

    B0 += (
        np.sum(hidden_delta, axis=0)
        * LEARNING_RATE
    )


    # --------------------------------------
    # 每 5000 次顯示 Loss
    # --------------------------------------

    if epoch % 5000 == 0:

        loss = np.mean(error ** 2)

        print(
            f"Epoch {epoch:6d} "
            f"Loss = {loss:.10f}"
        )


# ==========================================================
# 17. 最終 Loss
# ==========================================================

_, _, _, prediction_norm = forward(X_norm)

loss = np.mean(
    (Y_norm - prediction_norm) ** 2
)

print("\n==========================================")
print("訓練完成")
print("==========================================")

print("Final Loss =", loss)


# ==========================================================
# 18. 顯示預測結果
# ==========================================================

prediction = (
    prediction_norm * Y_range
    + Y_min
)

print("\n==========================================")
print("實際 Target / 預測 Output")
print("==========================================")

result = pd.DataFrame(
    np.hstack((Y, prediction)),
    columns=[
        "Target1",
        "Target2",
        "Target3",
        "Predict1",
        "Predict2",
        "Predict3"
    ]
)

display(result.head(20))


# ==========================================================
# 19. 顯示 W0
# ==========================================================

print("\n==========================================")
print("W0  (3 × 13)")
print("==========================================")

print(W0)


# ==========================================================
# 20. 顯示 B0
# ==========================================================

print("\n==========================================")
print("B0  (13)")
print("==========================================")

print(B0)


# ==========================================================
# 21. 顯示 W1
# ==========================================================

print("\n==========================================")
print("W1  (13 × 3)")
print("==========================================")

print(W1)


# ==========================================================
# 22. 顯示 B1
# ==========================================================

print("\n==========================================")
print("B1  (3)")
print("==========================================")

print(B1)


# ==========================================================
# 23. 輸出 ESP32 C++ 格式
# ==========================================================

def print_cpp_matrix(name, matrix):

    rows, cols = matrix.shape

    print(f"float {name}[{rows}][{cols}] = {{")

    for row in matrix:

        print(
            "    {"
            + ", ".join(f"{v:.10f}" for v in row)
            + "},"
        )

    print("};")
    print()


def print_cpp_array(name, array):

    print(
        f"float {name}[{len(array)}] = {{"
    )

    print(
        "    "
        + ", ".join(f"{v:.10f}" for v in array)
    )

    print("};")
    print()


print("\n")
print("==========================================================")
print("ESP32 C++ WEIGHTS")
print("==========================================================")

print_cpp_matrix("W0", W0)

print_cpp_array("B0", B0)

print_cpp_matrix("W1", W1)

print_cpp_array("B1", B1)


# ==========================================================
# 24. 正規化參數
#
# ESP32 如果要使用相同的神經網路，
# 必須同時使用這些參數。
# ==========================================================

print("\n")
print("==========================================================")
print("ESP32 NORMALIZATION PARAMETERS")
print("==========================================================")

print_cpp_array("X_MIN", X_min)

print_cpp_array("X_MAX", X_max)

print_cpp_array("Y_MIN", Y_min)

print_cpp_array("Y_MAX", Y_max)